# California Residential Home Price Prediction
# XGBoost Model

This notebook trains an XGBoost regressor, the primary advanced model for this project, using the fully enriched feature set including school district and geographic clustering.

**Evaluation metrics:** R2, MAPE, MdAPE

In [13]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error
from sklearn.model_selection import train_test_split

## Section 1 - Setup and Data Loading

In [2]:
df_model = pd.read_csv('data/cleaned_data_final.csv')
print('Shape:', df_model.shape)

Shape: (397461, 24)


## Section 2 - Train Test Split

Same temporal split approach used throughout this project, most recent month as test, all preceding months as training.

In [3]:
latest_month = df_model['CloseYear'] * 100 + df_model['CloseMonth']
test_month = latest_month.max()
print(f'Test month: {test_month}')

df_train = df_model[latest_month != test_month].copy()
df_test = df_model[latest_month == test_month].copy()

X_train = df_train.drop(columns=['ClosePrice'])
y_train = df_train['ClosePrice']
X_test = df_test.drop(columns=['ClosePrice'])
y_test = df_test['ClosePrice']

print(f'Train: {X_train.shape}\nTest: {X_test.shape}')

Test month: 202606
Train: (384874, 23)
Test: (12587, 23)


## Section 3 - Missing Values

Unlike Linear Regression and Random Forest, XGBoost handles missing values natively. BedBathRatio's 143 missing values are left as is, since XGBoost learns the optimal split direction for missing data automatically, rather than requiring manual imputation.

## Section 4 - Model Training, Version A

Version A is a first pass XGBoost model using default hyperparameters, establishing a baseline before any tuning.

In [4]:
model_a = xgb.XGBRegressor(random_state=304, n_jobs=-1)
model_a.fit(X_train, y_train)

y_pred_a = model_a.predict(X_test)
print('Version A trained.')

Version A trained.


### Evaluation, Version A

In [5]:
r2_a = r2_score(y_test, y_pred_a)
mape_a = mean_absolute_percentage_error(y_test, y_pred_a)
mdape_a = np.median(np.abs((y_test - y_pred_a) / y_test))

print(f'R2: {r2_a:.4f}')
print(f'MAPE: {mape_a:.4f}')
print(f'MdAPE: {mdape_a:.4f}')

R2: 0.8851
MAPE: 0.1547
MdAPE: 0.0997


### Summary, Version A

Version A results on the held out test month:

- R2: 0.8851
- MAPE: 0.1547
- MdAPE: 0.0997

This initial default configuration underperforms Random Forest, R2 0.9040, MdAPE 0.0733. This is expected for an untuned baseline. Hyperparameter tuning in version B is expected to close this gap and surpass Random Forest, since XGBoost with appropriate tuning typically outperforms bagged tree ensembles on structured tabular data.

## Section 5 - Model Training, Version B

Version B tunes key hyperparameters based on the results of version A. Max depth is increased to better capture interactions among high cardinality categorical features like City and PostalCode, the learning rate is raised to 0.1 with a larger number of estimators to compensate, and row and column subsampling are added to improve generalization on the held out test month.

In [6]:
model_b = xgb.XGBRegressor(
    n_estimators=800,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=304,
    n_jobs=-1
)
model_b.fit(X_train, y_train)

y_pred_b = model_b.predict(X_test)
print('Version B trained.')

Version B trained.


### Evaluation, Version B

In [7]:
r2_b = r2_score(y_test, y_pred_b)
mape_b = mean_absolute_percentage_error(y_test, y_pred_b)
mdape_b = np.median(np.abs((y_test - y_pred_b) / y_test))

print(f'R2: {r2_b:.4f}')
print(f'MAPE: {mape_b:.4f}')
print(f'MdAPE: {mdape_b:.4f}')

R2: 0.9138
MAPE: 0.1236
MdAPE: 0.0752


### Summary, Version B

Version B results on the held out test month:

- R2: 0.9138
- MAPE: 0.1236
- MdAPE: 0.0752

This is a meaningful improvement over version A, and now slightly exceeds Random Forest on R2 and MAPE, with MdAPE essentially matching. Increasing tree depth allowed the model to better capture interactions among high cardinality categorical features, while subsampling improved generalization to the unseen test month.

## Section 6 - Model Training, Version C

## Part 1 - Training Window

Version C begins by testing the training window length as a tunable parameter, per the task prompt guidance. Rather than always using all available historical data, different window lengths are tested to see whether more recent, potentially more representative data outperforms using the full history.

In [8]:
df_model['CloseDateActual'] = pd.to_datetime(df_model['CloseYear'].astype(str) + '-' + df_model['CloseMonth'].astype(str) + '-01')
test_date = df_model[latest_month == test_month]['CloseDateActual'].iloc[0]

window_lengths = [3, 6, 12, 24, None]
results = []

for window in window_lengths:
    if window is None:
        train_data = df_model[latest_month != test_month].copy()
        label = 'All available data'
    else:
        cutoff_date = test_date - pd.DateOffset(months=window)
        train_data = df_model[(latest_month != test_month) & (df_model['CloseDateActual'] > cutoff_date)].copy()
        label = f'{window} months'
    
    X_train_w = train_data.drop(columns=['ClosePrice', 'CloseDateActual'])
    y_train_w = train_data['ClosePrice']
    
    model_w = xgb.XGBRegressor(
        n_estimators=800, max_depth=8, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8,
        random_state=304, n_jobs=-1
    )
    model_w.fit(X_train_w, y_train_w)
    y_pred_w = model_w.predict(X_test.drop(columns=['CloseDateActual'], errors='ignore'))
    
    r2_w = r2_score(y_test, y_pred_w)
    mdape_w = np.median(np.abs((y_test - y_pred_w) / y_test))
    
    results.append({'window': label, 'train_rows': len(train_data), 'r2': r2_w, 'mdape': mdape_w})

results_df = pd.DataFrame(results)
print(results_df)

               window  train_rows        r2     mdape
0            3 months       23555  0.880484  0.088793
1            6 months       50234  0.893078  0.081964
2           12 months      116277  0.906376  0.078305
3           24 months      247161  0.912492  0.076190
4  All available data      384874  0.913759  0.075184


### Training Window Findings

Testing window lengths of 3, 6, 12, 24 months, and all available data shows a consistent improvement in both R2 and MdAPE as more historical training data is included, with no point at which additional older data begins to hurt performance. This suggests the underlying pricing relationships captured by the model, even across the 2022 to 2023 volatility identified in EDA, remain useful signal rather than becoming misleading as time passes. All available training data is used going forward as the optimal choice.

## Part 2 - Log Transforming the Target

EDA identified ClosePrice as heavily right skewed. Training on the log transformed target often helps tree based models produce more balanced errors across price ranges, since the model optimizes squared error on a scale where extreme high priced homes no longer dominate the loss function.

In [9]:
y_train_log = np.log1p(y_train)

model_log = xgb.XGBRegressor(
    n_estimators=800, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    random_state=304, n_jobs=-1
)
model_log.fit(X_train, y_train_log)

y_pred_log_raw = model_log.predict(X_test)
y_pred_log = np.expm1(y_pred_log_raw)

r2_log = r2_score(y_test, y_pred_log)
mape_log = mean_absolute_percentage_error(y_test, y_pred_log)
mdape_log = np.median(np.abs((y_test - y_pred_log) / y_test))

print(f'R2: {r2_log:.4f}')
print(f'MAPE: {mape_log:.4f}')
print(f'MdAPE: {mdape_log:.4f}')

R2: 0.9130
MAPE: 0.1156
MdAPE: 0.0727


### Log Transform Findings

Training on log transformed ClosePrice keeps R2 essentially unchanged, 0.9130 versus 0.9138, but improves MAPE from 0.1236 to 0.1156 and MdAPE from 0.0752 to 0.0727. This matches the expectation from EDA, where ClosePrice was found to be heavily right skewed. Log transforming the target reduces the influence of extreme high value properties on the loss function, producing more balanced percentage errors across the price range.  
  
This transformation is carried forward into the final model.

## Part 3 - Monotonic Constraints

Monotonic constraints force certain features to only move predictions in one logical direction. For example, increasing LivingArea should never decrease predicted price, all else equal. This improves the business logic and trustworthiness of the model, beyond just optimizing raw error metrics.

In [10]:
feature_names = X_train.columns.tolist()

monotone_constraints = {}
for col in feature_names:
    if col == 'LivingArea':
        monotone_constraints[col] = 1
    elif col == 'PropertyAge':
        monotone_constraints[col] = -1
    else:
        monotone_constraints[col] = 0

constraints_tuple = tuple(monotone_constraints[col] for col in feature_names)

In [11]:
model_mono = xgb.XGBRegressor(
    n_estimators=800, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    monotone_constraints=constraints_tuple,
    random_state=304, n_jobs=-1
)
model_mono.fit(X_train, y_train_log)

y_pred_mono_raw = model_mono.predict(X_test)
y_pred_mono = np.expm1(y_pred_mono_raw)

r2_mono = r2_score(y_test, y_pred_mono)
mape_mono = mean_absolute_percentage_error(y_test, y_pred_mono)
mdape_mono = np.median(np.abs((y_test - y_pred_mono) / y_test))

print(f'R2: {r2_mono:.4f}')
print(f'MAPE: {mape_mono:.4f}')
print(f'MdAPE: {mdape_mono:.4f}')

R2: 0.9045
MAPE: 0.1196
MdAPE: 0.0773


### Monotonic Constraints Findings

Applying monotonic constraints to LivingArea and PropertyAge, forcing price to increase with square footage and decrease with age, slightly reduced performance across all three metrics compared to the unconstrained log transformed model, R2 dropped from 0.9130 to 0.9045, and both MAPE and MdAPE increased slightly. This reflects a real tradeoff, monotonic constraints improve interpretability and enforce intuitive business logic, but restrict the model's ability to capture legitimate exceptions in the data, such as renovated older properties. Given the accuracy cost outweighs the interpretability benefit here, constraints are not used in the final model.

## Part 4 - Regularization Tuning

L1 and L2 regularization penalize overly complex trees, which can improve generalization to unseen data by discouraging the model from fitting noise. This is tested on top of the log transformed target, since that was the best performing configuration so far.

In [12]:
model_reg = xgb.XGBRegressor(
    n_estimators=800, max_depth=8, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8,
    reg_alpha=1.0, reg_lambda=2.0,
    random_state=304, n_jobs=-1
)
model_reg.fit(X_train, y_train_log)

y_pred_reg_raw = model_reg.predict(X_test)
y_pred_reg = np.expm1(y_pred_reg_raw)

r2_reg = r2_score(y_test, y_pred_reg)
mape_reg = mean_absolute_percentage_error(y_test, y_pred_reg)
mdape_reg = np.median(np.abs((y_test - y_pred_reg) / y_test))

print(f'R2: {r2_reg:.4f}')
print(f'MAPE: {mape_reg:.4f}')
print(f'MdAPE: {mdape_reg:.4f}')

R2: 0.9123
MAPE: 0.1158
MdAPE: 0.0717


### Regularization Findings

Adding L1 and L2 regularization on top of the log transformed model produced negligible change across all metrics, R2 0.9123 versus 0.9130, MAPE 0.1158 versus 0.1156, MdAPE 0.0717 versus 0.0727. This suggests the model was not significantly overfitting to begin with, likely because subsample and colsample_bytree introduced in version B already provided sufficient regularization. Additional regularization is not included in the final model, since it adds complexity without measurable benefit.

## Part 5 - Early Stopping

Rather than fixing the number of trees in advance, early stopping monitors performance on a validation set during training and stops once further trees stop improving results, automatically finding an efficient number of estimators instead of guessing one.

In [ ]:
X_train_sub, X_val, y_train_sub, y_val = train_test_split(X_train, y_train_log, test_size=0.1, random_state=304)

model_early = xgb.XGBRegressor(
    n_estimators=2000,
    max_depth=8,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=304,
    n_jobs=-1,
    early_stopping_rounds=20,
    eval_metric='rmse'
)

model_early.fit(X_train_sub, y_train_sub, eval_set=[(X_val, y_val)], verbose=False)

print('Best iteration:', model_early.best_iteration)

y_pred_early_raw = model_early.predict(X_test)
y_pred_early = np.expm1(y_pred_early_raw)

r2_early = r2_score(y_test, y_pred_early)
mape_early = mean_absolute_percentage_error(y_test, y_pred_early)
mdape_early = np.median(np.abs((y_test - y_pred_early) / y_test))

print(f'R2: {r2_early:.4f}')
print(f'MAPE: {mape_early:.4f}')
print(f'MdAPE: {mdape_early:.4f}')

Best iteration: 1078
R2: 0.9138
MAPE: 0.1143
MdAPE: 0.0715


### Early Stopping Findings

Using early stopping with a held out validation set, the model automatically determined 1078 as the optimal number of trees, improving over the manually chosen 800 estimators used in earlier versions. This produced the best results so far, R2 0.9138, MAPE 0.1143, MdAPE 0.0715, slightly outperforming the plain log transformed model across MAPE and MdAPE while matching it on R2. This confirms that letting the model determine tree count based on validation performance is more effective than a fixed, manually chosen value.

## Part 6 - Randomized Hyperparameter Search

A systematic search across a range of hyperparameter combinations, to check whether further tuning beyond the manually chosen values in earlier versions can improve on the current best result.

In [15]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [600, 800, 1000, 1200],
    'max_depth': [6, 8, 10],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9]
}

search = RandomizedSearchCV(
    xgb.XGBRegressor(random_state=304, n_jobs=-1),
    param_distributions=param_dist,
    n_iter=15,
    scoring='r2',
    cv=3,
    random_state=304,
    n_jobs=-1
)

search.fit(X_train, y_train_log)
print('Best params:', search.best_params_)

y_pred_search_raw = search.best_estimator_.predict(X_test)
y_pred_search = np.expm1(y_pred_search_raw)

r2_search = r2_score(y_test, y_pred_search)
mape_search = mean_absolute_percentage_error(y_test, y_pred_search)
mdape_search = np.median(np.abs((y_test - y_pred_search) / y_test))

print(f'R2: {r2_search:.4f}')
print(f'MAPE: {mape_search:.4f}')
print(f'MdAPE: {mdape_search:.4f}')

KeyboardInterrupt: 